# Nghiên Cứu & Huấn Luyện Mô Hình Transformer Nhận Diện Té Ngã Dựa Trên Chuỗi Tư Thế Không Gian - Thời Gian (Spatial-Temporal Skeleton Sequence)

**Bài toán nghiên cứu:** Nhận diện chính xác hành vi té ngã (Fall Detection) từ chuỗi 30 khung hình liên tiếp của 17 điểm khớp xương cơ thể 3D trích xuất bởi MediaPipe BlazePose.

---

### 1. Bối cảnh & Mục tiêu Khoa học
Trong các hệ thống giám sát sức khỏe thông minh và hỗ trợ người cao tuổi (AAL - Ambient Assisted Living), té ngã là một trong những nguyên nhân hàng đầu gây chấn thương nghiêm trọng. Nhận diện kịp thời cú ngã giúp giảm thiểu thời gian nằm bất động và tăng khả năng cứu chữa.

So với các hướng tiếp cận truyền thống:
* **Phương pháp dùng trực tiếp Video (RGB Image Sequences):** Tiêu tốn băng thông tính toán rất lớn, chịu ảnh hưởng mạnh bởi điều kiện ánh sáng, góc quay camera và đặc biệt là vi phạm quyền riêng tư (Privacy Concerns) của người sử dụng trong không gian riêng tư (phòng ngủ, phòng tắm).
* **Phương pháp RNN / LSTM cổ điển trên chuỗi khớp xương:** Thường gặp hiện tượng suy giảm gradient (vanishing gradient) khi mô hình hóa các chuỗi dài, đồng thời khó nắm bắt được mối quan hệ không gian tương hỗ phức tạp giữa các khớp xương xa nhau trong cùng một khung hình.

**Mục tiêu của nghiên cứu này:**
1. Thiết kế và huấn luyện kiến trúc **Transformer Encoder** chuyên biệt cho chuỗi thời gian không gian (Spatial-Temporal Skeleton Modeling), tận dụng cơ chế **Self-Attention** đa đầu (Multi-Head Attention) để học các đặc trưng biến đổi tư thế đột ngột.
2. Triển khai kỹ thuật **Tăng Cường Dữ Liệu Chuỗi Khung Xương (Spatial-Temporal Data Augmentation)** trên tọa độ thực nghiệm (xoay quanh trọng tâm, dịch chuyển, co giãn tỉ lệ, thêm nhiễu vị trí Gaussian jitter) trước khi chuẩn hóa, giúp mô hình có khả năng tổng quát hóa cao và chống hiện tượng học vẹt (overfitting).
3. Chuẩn hóa không gian bất biến theo vị trí và khoảng cách camera thông qua kỹ thuật dịch tâm hông (Mid-Hip centering) và chuẩn hóa kích thước cơ thể (Torso scaling).
4. Áp dụng chiến lược huấn luyện với lịch học suy giảm đa thức (**Polynomial Decay Learning Rate Schedule**), dừng sớm (**EarlyStopping**) dựa trên chỉ số **F1-Macro**, và tối ưu hóa ngưỡng kích hoạt (**Optimal Threshold Search**) nhằm giảm thiểu tối đa hiện tượng bỏ sót ca ngã (False Negatives).
5. Đóng gói và chuyển đổi mô hình sang định dạng **TensorFlow Lite (TFLite)** siêu nhẹ (~313 KB), sẵn sàng triển khai trên các thiết bị nhúng IoT (Raspberry Pi 4 / Jetson Nano) với độ trễ suy luận thời gian thực (< 15 ms).

## 1. Cấu Hình Tham Số & Đường Dẫn Thí Nghiệm (Configuration & Hyperparameters)

Tất cả các siêu tham số của mô hình, cài đặt tăng cường dữ liệu và đường dẫn tệp tin được quản lý tập trung trong một cấu trúc cấu hình đồng nhất. Điều này đảm bảo tính minh bạch, khả năng tái lập và hỗ trợ chạy linh hoạt trên các môi trường khác nhau (Kaggle Notebooks, Google Colab hoặc Máy chủ Cục bộ).

In [ ]:
# ===== THIẾT LẬP CẤU HÌNH THÍ NGHIỆM TẬP TRUNG =====
import os
import json
from pathlib import Path

# Tự động nhận diện thư mục đầu ra (Output Directory): Trên Kaggle luôn là /kaggle/working
OUTPUT_DIR = Path('/kaggle/working') if os.path.exists('/kaggle/working') else Path('.')

CONFIG = {
    # 1. Đường dẫn dữ liệu và thư mục xuất kết quả (Output)
    'paths': {
        'kaggle_train': '/kaggle/input/fall-dataset6/train',
        'kaggle_val':   '/kaggle/input/fall-dataset6/val',
        'kaggle_test':  '/kaggle/input/fall-dataset6/test',
        'local_train':  '../data/train',
        'local_val':    '../data/val',
        'local_test':   '../data/test',
        'output_dir':   str(OUTPUT_DIR),
        'export_saved_model': str(OUTPUT_DIR / 'fall_model_exported_sm'),
        'export_tflite':      str(OUTPUT_DIR / 'fall_detection_transformer.tflite'),
        'csv_error_log':      str(OUTPUT_DIR / 'error_analysis_report.csv')
    },
    
    # 2. Đặc trưng đầu vào khung xương (Skeleton Dimensions)
    'skeleton': {
        'timesteps': 30,             # Độ dài cửa sổ trượt: 30 khung hình (~ 1 giây ở 30 FPS)
        'num_keypoints': 17,         # 17 điểm khớp trích xuất từ MediaPipe BlazePose
        'channels_per_keypoint': 3,  # Tọa độ (x, y, confidence/z)
        'min_confidence_norm': 0.2   # Ngưỡng tin cậy tối thiểu khi chuẩn hóa khớp
    },
    
    # 3. Tăng cường dữ liệu (Data Augmentation)
    'augmentation': {
        'enable_scaling': False,
        'enable_rotation': True,
        'enable_translation': True,
        'enable_jitter': True,
        'enable_confidence_noise': False,
        'enable_dropout': False,
        'augment_counts': {0: 1, 1: 1} # Tỉ lệ nhân bản: {no_fall: +100%, fall: +100%}
    },
    
    # 4. Kiến trúc mạng Transformer Encoder
    'model': {
        'num_encoder_blocks': 3,     # Số lượng khối Transformer Encoder
        'd_model': 64,               # Kích thước không gian đặc trưng ẩn
        'num_heads': 4,              # Số lượng đầu chú ý (Attention Heads)
        'ff_dim': 128,               # Kích thước tầng Feed-Forward (d_model * 2)
        'projection_dim': 64,        # Kích thước chiếu đặc trưng ban đầu
        'final_dense_units': 32,     # Số nơ-ron tầng Dense phân loại ẩn
        'dropout_rate': 0.06         # Tỉ lệ Dropout chống quá khớp
    },
    
    # 5. Huấn luyện & Đánh giá chéo (Training & Cross-Validation)
    'training': {
        'seed': 42,
        'cv_splits': 5,              # 5-Fold Stratified Cross-Validation
        'cv_epochs': 80,
        'cv_batch_size': 32,
        'cv_patience': 10,
        'final_epochs': 80,
        'final_batch_size': 32,
        'final_patience': 15,
        'initial_learning_rate': 0.0001,
        'end_learning_rate': 0.00005,
        'default_threshold': 0.50
    }
}

# Tính toán kích thước vector đầu vào của một khung hình và toàn chuỗi
CONFIG['skeleton']['features_per_frame'] = (
    CONFIG['skeleton']['num_keypoints'] * CONFIG['skeleton']['channels_per_keypoint']
)
CONFIG['skeleton']['input_shape'] = (
    CONFIG['skeleton']['timesteps'],
    CONFIG['skeleton']['features_per_frame']
)

print('[Cấu Hình] Thiết lập tham số thành công:')
print(f'Thư mục xuất kết quả (OUTPUT_DIR): {OUTPUT_DIR}')
print(json.dumps(CONFIG, indent=2))

## 2. Kiểm Tra Môi Trường Phần Cứng & Khởi Tạo Tính Tái Lập (Hardware Preflight & Determinism)

Để đảm bảo tính khách quan và khoa học, toàn bộ mã nguồn thực nghiệm cần đáp ứng:
1. Xác định sự sẵn có của bộ tăng tốc đồ họa (GPU / CUDA) để tăng tốc độ tính toán.
2. Cố định hạt giống ngẫu nhiên (**Deterministic Random Seed**) trên mọi thư viện cốt lõi (`os`, `random`, `numpy`, `tensorflow`) nhằm tái lập 100% kết quả huấn luyện.

In [ ]:
import os
import sys
import random
import time
import numpy as np
import tensorflow as tf

# 1. Kiểm tra môi trường phần cứng và tăng tốc GPU
print("=" * 60)
print(f"Phiên bản Python     : {sys.version.split()[0]}")
print(f"Phiên bản TensorFlow : {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        print(f"Thiết bị GPU         : {gpu.name}")
    try:
        # Bật cơ chế cấp phát bộ nhớ động (Dynamic Memory Growth) cho GPU
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Trạng thái GPU       : Đã kích hoạt bộ nhớ tăng trưởng động (Memory Growth).")
    except Exception as e:
        print(f"Lưu ý cấu hình GPU   : {e}")
else:
    print("Trạng thái GPU       : Không tìm thấy GPU, hệ thống sẽ chạy trên CPU.")
print("=" * 60)

# 2. Thiết lập cơ chế ngẫu nhiên tất định (Reproducibility Seed)
def set_deterministic_seed(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    print(f"[Khởi tạo] Đã cố định Random Seed toàn hệ thống = {seed}")

set_deterministic_seed(CONFIG['training']['seed'])

## 3. Khai Báo 17 Điểm Khớp & Thuật Toán Chuẩn Hóa Khung Xương (Skeleton Normalization)

Mô hình sử dụng 17 điểm khớp xương chuẩn trích xuất từ **MediaPipe BlazePose**:
* **Vùng mặt (5 khớp):** `Nose`, `Left Eye`, `Right Eye`, `Left Ear`, `Right Ear`.
* **Thân trên (6 khớp):** `Left Shoulder`, `Right Shoulder`, `Left Elbow`, `Right Elbow`, `Left Wrist`, `Right Wrist`.
* **Thân dưới (6 khớp):** `Left Hip`, `Right Hip`, `Left Knee`, `Right Knee`, `Left Ankle`, `Right Ankle`.

### Thuật toán Chuẩn Hóa Không Gian Bất Biến (Spatial Invariance Normalization):
Để triệt tiêu ảnh hưởng của vị trí người đứng trong khung hình và khoảng cách xa/gần so với camera, mỗi khung hình tại thời điểm $t$ được chuẩn hóa qua 2 bước:
1. **Dịch chuyển gốc tọa độ về Trung Điểm Hông (Centroid Translation):**
   $$	ext{Mid-Hip} = \left(rac{x_{	ext{Left Hip}} + x_{	ext{Right Hip}}}{2}, rac{y_{	ext{Left Hip}} + y_{	ext{Right Hip}}}{2}ight)$$
   $$x'_i = x_i - 	ext{Mid-Hip}_x, \quad y'_i = y_i - 	ext{Mid-Hip}_y$$
2. **Chuẩn hóa kích thước cơ thể theo chiều dài Thân (Torso Scaling):**
   $$	ext{Mid-Shoulder} = \left(rac{x_{	ext{Left Shoulder}} + x_{	ext{Right Shoulder}}}{2}, rac{y_{	ext{Left Shoulder}} + y_{	ext{Right Shoulder}}}{2}ight)$$
   $$H_{	ext{torso}} = |	ext{Mid-Shoulder}_y - 	ext{Mid-Hip}_y|$$
   $$x''_i = rac{x'_i}{H_{	ext{torso}}}, \quad y''_i = rac{y'_i}{H_{	ext{torso}}}$$
*(Độ tin cậy $c_i$ hoặc tọa độ $z_i$ được giữ nguyên vẹn).*

In [ ]:
# ===== 1. ĐỊNH NGHĨA DANH SÁCH KHỚP VÀ CHỈ SỐ TRA CỨU =====
KEYPOINT_NAMES = [
    'Nose', 'Left Eye', 'Right Eye', 'Left Ear', 'Right Ear',
    'Left Shoulder', 'Right Shoulder', 'Left Elbow', 'Right Elbow',
    'Left Wrist', 'Right Wrist', 'Left Hip', 'Right Hip',
    'Left Knee', 'Right Knee', 'Left Ankle', 'Right Ankle'
]
# Sắp xếp theo thứ tự bảng chữ cái để đảm bảo tính nhất quán giữa khâu nạp và suy luận
SORTED_KEYPOINT_NAMES = sorted(KEYPOINT_NAMES)
KEYPOINT_DICT = {name: i for i, name in enumerate(SORTED_KEYPOINT_NAMES)}
NUM_KEYPOINTS = len(KEYPOINT_DICT)

print(f"[Khớp xương] Tổng số khớp chuẩn hóa: {NUM_KEYPOINTS}")
print(f"[Thứ tự sắp xếp]: {SORTED_KEYPOINT_NAMES}")

def get_kpt_indices(keypoint_name):
    """
    Trả về chỉ số bắt đầu của khớp trong vector đặc trưng (x_idx, y_idx, c_idx).
    """
    if keypoint_name not in KEYPOINT_DICT:
        available_keys = list(KEYPOINT_DICT.keys())
        similar_keys = [k for k in available_keys if keypoint_name.lower() in k.lower()]
        raise ValueError(f"Khớp '{keypoint_name}' không tồn tại. Gợi ý: {similar_keys}")
    kp_idx = KEYPOINT_DICT[keypoint_name]
    return kp_idx * 3, kp_idx * 3 + 1, kp_idx * 3 + 2


# ===== 2. HÀM CHUẨN HÓA KHUNG XƯƠNG KHÔNG GIAN =====
def normalize_skeleton(sequence_data, min_confidence=0.2):
    """
    Chuẩn hóa chuỗi khung xương:
    1. Dịch tâm về trung điểm hông (Mid-Hip).
    2. Co giãn theo chiều dài thân (Torso Length = khoảng cách giữa Mid-Shoulder và Mid-Hip).
    
    Tham số:
        sequence_data (np.ndarray): Mảng numpy kích thước (30, 51)
        min_confidence (float): Độ tin cậy tối thiểu để sử dụng khớp làm mốc quy chiếu
    Trả về:
        np.ndarray: Chuỗi khung xương đã được chuẩn hóa
    """
    normalized_sequence = np.copy(sequence_data)
    ref_kp_names = {
        'ls': 'Left Shoulder', 'rs': 'Right Shoulder',
        'lh': 'Left Hip',      'rh': 'Right Hip'
    }

    for t in range(sequence_data.shape[0]):
        frame_data = sequence_data[t, :]
        current_frame_normalized = np.copy(frame_data)
        
        try:
            ls_x_idx, ls_y_idx, ls_c_idx = get_kpt_indices(ref_kp_names['ls'])
            rs_x_idx, rs_y_idx, rs_c_idx = get_kpt_indices(ref_kp_names['rs'])
            lh_x_idx, lh_y_idx, lh_c_idx = get_kpt_indices(ref_kp_names['lh'])
            rh_x_idx, rh_y_idx, rh_c_idx = get_kpt_indices(ref_kp_names['rh'])
        except ValueError:
            normalized_sequence[t, :] = frame_data
            continue

        ls_x, ls_y, ls_c = frame_data[ls_x_idx], frame_data[ls_y_idx], frame_data[ls_c_idx]
        rs_x, rs_y, rs_c = frame_data[rs_x_idx], frame_data[rs_y_idx], frame_data[rs_c_idx]
        lh_x, lh_y, lh_c = frame_data[lh_x_idx], frame_data[lh_y_idx], frame_data[lh_c_idx]
        rh_x, rh_y, rh_c = frame_data[rh_x_idx], frame_data[rh_y_idx], frame_data[rh_c_idx]

        # 1. Xác định tọa độ Trung Điểm Vai (Mid-Shoulder)
        mid_shoulder_x, mid_shoulder_y = np.nan, np.nan
        valid_ls, valid_rs = ls_c > min_confidence, rs_c > min_confidence
        if valid_ls and valid_rs:
            mid_shoulder_x, mid_shoulder_y = (ls_x + rs_x) / 2, (ls_y + rs_y) / 2
        elif valid_ls:
            mid_shoulder_x, mid_shoulder_y = ls_x, ls_y
        elif valid_rs:
            mid_shoulder_x, mid_shoulder_y = rs_x, rs_y

        # 2. Xác định tọa độ Trung Điểm Hông (Mid-Hip)
        mid_hip_x, mid_hip_y = np.nan, np.nan
        valid_lh, valid_rh = lh_c > min_confidence, rh_c > min_confidence
        if valid_lh and valid_rh:
            mid_hip_x, mid_hip_y = (lh_x + rh_x) / 2, (lh_y + rh_y) / 2
        elif valid_lh:
            mid_hip_x, mid_hip_y = lh_x, lh_y
        elif valid_rh:
            mid_hip_x, mid_hip_y = rh_x, rh_y

        # Nếu không xác định được hông, giữ nguyên khung hình đó
        if np.isnan(mid_hip_x) or np.isnan(mid_hip_y):
            normalized_sequence[t, :] = frame_data
            continue

        # 3. Tính chiều dài thân người (Torso Height)
        reference_height = np.nan
        if not np.isnan(mid_shoulder_y) and not np.isnan(mid_hip_y):
            reference_height = np.abs(mid_shoulder_y - mid_hip_y)

        perform_scaling = not (np.isnan(reference_height) or reference_height <= 1e-6)

        # 4. Áp dụng chuẩn hóa cho toàn bộ 17 khớp
        for kp_name, kp_idx in KEYPOINT_DICT.items():
            x_idx, y_idx, c_idx = kp_idx * 3, kp_idx * 3 + 1, kp_idx * 3 + 2
            raw_x, raw_y = frame_data[x_idx], frame_data[y_idx]

            shifted_x = raw_x - mid_hip_x
            shifted_y = raw_y - mid_hip_y

            if perform_scaling:
                current_frame_normalized[x_idx] = shifted_x / reference_height
                current_frame_normalized[y_idx] = shifted_y / reference_height
            else:
                current_frame_normalized[x_idx] = shifted_x
                current_frame_normalized[y_idx] = shifted_y
            
            # Giữ nguyên độ tin cậy c
            current_frame_normalized[c_idx] = frame_data[c_idx]

        normalized_sequence[t, :] = current_frame_normalized

    return normalized_sequence

## 4. Tăng Cường Dữ Liệu Chuỗi Thời Gian - Không Gian (Spatial-Temporal Data Augmentation)

Để mô hình học được các đặc trưng bất biến trước các biến thiên vật lý thực tế của camera giám sát trong gia đình, chúng ta áp dụng các phép biến đổi hình học **nhất quán trên toàn chuỗi 30 frames** trước khi thực hiện chuẩn hóa:
1. **Xoay khung xương ngẫu nhiên quanh tâm hông (Rotation Augmentation):**
   Góc xoay $\theta \sim \mathcal{U}(-15^\circ, 15^\circ)$ mô phỏng góc nghiêng lắp đặt của camera hoặc tư thế ngã chéo góc.
2. **Dịch chuyển ngẫu nhiên toàn thân (Translation Augmentation):**
   Độ dịch $\Delta x, \Delta y \sim \mathcal{U}(-0.05, 0.05)$ mô phỏng người chuyển động qua lại trong phòng.
3. **Thêm nhiễu Gaussian vị trí (Jittering Augmentation):**
   Nhiễu $\epsilon \sim \mathcal{N}(0, 0.01)$ mô phỏng hiện tượng rung nhiễu của thuật toán ước lượng tư thế hoặc rung lắc nhẹ của camera.
4. **Co giãn tỉ lệ (Scaling Augmentation):**
   Tỉ lệ $s \sim \mathcal{U}(0.9, 1.1)$ mô phỏng vóc dáng cơ thể cao gầy / thấp đậm khác nhau.

In [ ]:
from glob import glob

# ===== 1. THUẬT TOÁN TĂNG CƯỜNG DỮ LIỆU CHUỖI KHUNG XƯƠNG =====
def augment_sequence(
    seq,
    enable_scaling=False,
    enable_rotation=True,
    enable_translation=True,
    enable_jitter=True,
    enable_confidence_noise=False,
    enable_dropout=False
):
    """
    Thực hiện tăng cường dữ liệu trên chuỗi tọa độ khớp xương thô (30, 51).
    Bảo toàn tính liên tục thời gian trên toàn bộ 30 khung hình.
    """
    seq = seq.copy()

    # 1. Co giãn tỉ lệ toàn bộ khung xương
    if enable_scaling:
        scale = np.random.uniform(0.9, 1.1)
        for i in range(17):
            seq[:, i * 3]     *= scale
            seq[:, i * 3 + 1] *= scale

    # 2. Xoay toàn bộ cơ thể quanh tâm hông một góc ngẫu nhiên [-15, +15] độ
    if enable_rotation:
        angle_deg = np.random.uniform(-15, 15)
        angle_rad = np.radians(angle_deg)
        cos_val = np.cos(angle_rad)
        sin_val = np.sin(angle_rad)

        lh_x_idx, lh_y_idx, _ = get_kpt_indices('Left Hip')
        rh_x_idx, rh_y_idx, _ = get_kpt_indices('Right Hip')

        lh_x, rh_x = seq[:, lh_x_idx], seq[:, rh_x_idx]
        lh_y, rh_y = seq[:, lh_y_idx], seq[:, rh_y_idx]
        center_x = (lh_x + rh_x) / 2
        center_y = (lh_y + rh_y) / 2

        for i in range(17):
            x = seq[:, i * 3]
            y = seq[:, i * 3 + 1]
            x_shifted = x - center_x
            y_shifted = y - center_y
            x_rot = x_shifted * cos_val - y_shifted * sin_val
            y_rot = x_shifted * sin_val + y_shifted * cos_val
            seq[:, i * 3]     = x_rot + center_x
            seq[:, i * 3 + 1] = y_rot + center_y

    # 3. Dịch chuyển ngẫu nhiên khung xương trong không gian
    if enable_translation:
        shift_x = np.random.uniform(-0.05, 0.05)
        shift_y = np.random.uniform(-0.05, 0.05)
        for i in range(17):
            seq[:, i * 3]     += shift_x
            seq[:, i * 3 + 1] += shift_y

    # 4. Thêm nhiễu vị trí Gaussian jittering mô phỏng rung nhiễu cảm biến
    if enable_jitter:
        noise = np.random.normal(0, 0.01, size=seq[:, :17 * 3].shape)
        seq[:, :17 * 3] += noise

    # 5. Nhiễu độ tin cậy cảm biến
    if enable_confidence_noise:
        for i in range(17):
            noise_c = np.random.normal(0, 0.05, size=seq[:, i * 3 + 2].shape)
            seq[:, i * 3 + 2] = np.clip(seq[:, i * 3 + 2] + noise_c, 0.0, 1.0)

    # 6. Dropout ngẫu nhiên một số khớp (mô phỏng bị che khuất)
    if enable_dropout:
        drop_rate = 0.05
        for i in range(17):
            if np.random.rand() < drop_rate:
                seq[:, i * 3]     = 0.0
                seq[:, i * 3 + 1] = 0.0
                seq[:, i * 3 + 2] = 0.0

    return seq


# ===== 2. HÀM TẢI TẬP DỮ LIỆU TỪ CÁC TỆP .NPY =====
def load_dataset(data_path, expected_shape_template=(30, None), normalize=True, min_confidence_norm=0.2):
    """
    Đọc các chuỗi dữ liệu khớp (.npy) từ các thư mục con: 'no_fall' (0) và 'fall' (1).
    """
    expected_shape = (expected_shape_template[0], NUM_KEYPOINTS * 3)
    X, y, npy_filepaths = [], [], []

    print(f"[Nạp dữ liệu] Thư mục: {data_path} (Kích thước yêu cầu: {expected_shape})")
    
    classes_config = [('no_fall', 0), ('fall', 1)]
    for label_name, label_int in classes_config:
        folder_path = os.path.join(data_path, label_name)
        if not os.path.exists(folder_path):
            print(f"  [Cảnh báo] Không tìm thấy thư mục: {folder_path}. Bỏ qua lớp '{label_name}'.")
            continue

        file_list = sorted(glob(os.path.join(folder_path, '*.npy')))
        print(f"  -> Lớp '{label_name}' (label={label_int}): Tìm thấy {len(file_list)} tệp tin.")

        for file_path in file_list:
            try:
                data = np.load(file_path)
                if data.shape == expected_shape:
                    if normalize:
                        data = normalize_skeleton(data, min_confidence=min_confidence_norm)
                    X.append(data)
                    y.append(label_int)
                    npy_filepaths.append(file_path)
            except Exception as e:
                print(f"  [Lỗi đọc tệp] {file_path}: {e}")

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.int32)
    return X, y, npy_filepaths

## 5. Pipeline Nạp & Tiền Xử Lý Dữ Liệu (Data Pipeline Execution)

**Quy trình chuẩn bị dữ liệu tuân thủ nghiêm ngặt:**
1. **Tập Huấn Luyện (Training Set):** Nạp dữ liệu thô (chưa chuẩn hóa) $\rightarrow$ Áp dụng Tăng cường dữ liệu (Augmentation) theo cấu hình $\rightarrow$ Chuẩn hóa toàn bộ tập dữ liệu đã tăng cường $\rightarrow$ Xáo trộn ngẫu nhiên (Shuffle).
2. **Tập Đánh Giá & Kiểm Thử (Validation & Test Sets):** Nạp và chuẩn hóa trực tiếp, tuyệt đối không áp dụng tăng cường dữ liệu để đảm bảo tính khách quan và ngăn chặn rò rỉ dữ liệu (Data Leakage).

In [ ]:
from sklearn.utils import shuffle

# 1. Xác định đường dẫn dữ liệu theo môi trường hiện tại
def resolve_data_paths(paths_cfg):
    # Kiểm tra xem đường dẫn Kaggle có khả dụng không
    if os.path.exists(paths_cfg['kaggle_train']):
        print("[Môi trường] Phát hiện môi trường Kaggle Dataset.")
        return paths_cfg['kaggle_train'], paths_cfg['kaggle_val'], paths_cfg['kaggle_test']
    elif os.path.exists(paths_cfg['local_train']):
        print("[Môi trường] Phát hiện môi trường Local Dataset.")
        return paths_cfg['local_train'], paths_cfg['local_val'], paths_cfg['local_test']
    else:
        # Fallback tạo thư mục mẫu hoặc báo lỗi rõ ràng
        print("[Môi trường] Dùng đường dẫn mặc định Kaggle (hoặc hãy trỏ đường dẫn dữ liệu của bạn).")
        return paths_cfg['kaggle_train'], paths_cfg['kaggle_val'], paths_cfg['kaggle_test']

base_train, base_val, base_test = resolve_data_paths(CONFIG['paths'])
EXPECTED_SHAPE_TEMPLATE = (CONFIG['skeleton']['timesteps'], None)
MIN_CONF_NORM = CONFIG['skeleton']['min_confidence_norm']

# 2. Nạp dữ liệu Train thô (để tiến hành Augmentation trước khi chuẩn hóa)
print("\n--- 1. Nạp Tập Huấn Luyện (Dữ liệu Thô để Tăng Cường) ---")
X_train_orig, y_train_orig, train_npy_filepaths = load_dataset(
    base_train, expected_shape_template=EXPECTED_SHAPE_TEMPLATE,
    normalize=False,
    min_confidence_norm=MIN_CONF_NORM
)

# 3. Nạp tập Validation và Test (Chuẩn hóa trực tiếp, không Augmentation)
print("\n--- 2. Nạp Tập Đánh Giá (Validation Set - Đã Chuẩn Hóa) ---")
X_val_orig, y_val_orig, val_npy_filepaths = load_dataset(
    base_val, expected_shape_template=EXPECTED_SHAPE_TEMPLATE,
    normalize=True,
    min_confidence_norm=MIN_CONF_NORM
)

print("\n--- 3. Nạp Tập Kiểm Thử (Test Set - Đã Chuẩn Hóa) ---")
X_test_orig, y_test_orig, test_npy_filepaths = load_dataset(
    base_test, expected_shape_template=EXPECTED_SHAPE_TEMPLATE,
    normalize=True,
    min_confidence_norm=MIN_CONF_NORM
)

# 4. Tiến hành Tăng Cường & Chuẩn Hóa Tập Huấn Luyện
if X_train_orig.size > 0:
    print(f"\n--- 4. Tiến Hành Tăng Cường & Chuẩn Hóa Dữ Liệu Huấn Luyện ---")
    print(f"Kích thước tập thô ban đầu: X = {X_train_orig.shape}, y = {y_train_orig.shape}")

    aug_cfg = CONFIG['augmentation']
    aug_counts = aug_cfg['augment_counts']
    aug_X_raw, aug_y = [], []

    for xi, yi in zip(X_train_orig, y_train_orig):
        count = aug_counts.get(int(yi), 1)
        for _ in range(count):
            augmented_sample = augment_sequence(
                xi,
                enable_scaling=aug_cfg['enable_scaling'],
                enable_rotation=aug_cfg['enable_rotation'],
                enable_translation=aug_cfg['enable_translation'],
                enable_jitter=aug_cfg['enable_jitter'],
                enable_confidence_noise=aug_cfg['enable_confidence_noise'],
                enable_dropout=aug_cfg['enable_dropout']
            )
            aug_X_raw.append(augmented_sample)
            aug_y.append(yi)

    # Gộp dữ liệu gốc và dữ liệu nhân bản
    X_train_augmented_raw = np.concatenate([X_train_orig, np.array(aug_X_raw, dtype=np.float32)], axis=0)
    y_train_augmented = np.concatenate([y_train_orig, np.array(aug_y, dtype=np.int32)], axis=0)
    print(f"Kích thước sau khi Augmentation (chưa chuẩn hóa): {X_train_augmented_raw.shape}")

    # Chuẩn hóa toàn bộ tập huấn luyện đã gộp
    print("Đang tiến hành chuẩn hóa toàn bộ tập huấn luyện...")
    X_train_normalized_list = [
        normalize_skeleton(seq, min_confidence=MIN_CONF_NORM)
        for seq in X_train_augmented_raw
    ]
    X_train_normalized = np.array(X_train_normalized_list, dtype=np.float32)

    # Xử lý các giá trị NaN phát sinh (nếu có)
    if np.isnan(X_train_normalized).any():
        X_train_normalized = np.nan_to_num(X_train_normalized, nan=0.0, posinf=0.0, neginf=0.0)
        print("  -> Đã tự động thay thế các giá trị NaN bằng 0.0.")

    # Xáo trộn dữ liệu (Shuffle)
    X_train_orig, y_train_orig = shuffle(
        X_train_normalized, y_train_augmented, random_state=CONFIG['training']['seed']
    )
    print(f"Kích thước tập huấn luyện hoàn chỉnh: {X_train_orig.shape}")

    # Giải phóng bộ nhớ đệm
    del X_train_augmented_raw, y_train_augmented, X_train_normalized, X_train_normalized_list, aug_X_raw, aug_y
else:
    print("[Lưu ý] Tập dữ liệu huấn luyện hiện tại đang rỗng hoặc chưa nạp được tệp .npy.")

print("\n" + "=" * 50)
print(f"TẬP DỮ LIỆU SẴN SÀNG CHO MÔ HÌNH:")
print(f"  * Train: X = {X_train_orig.shape if X_train_orig.size > 0 else '0'}, y = {y_train_orig.shape if y_train_orig.size > 0 else '0'}")
print(f"  * Val  : X = {X_val_orig.shape if X_val_orig.size > 0 else '0'}, y = {y_val_orig.shape if y_val_orig.size > 0 else '0'}")
print(f"  * Test : X = {X_test_orig.shape if X_test_orig.size > 0 else '0'}, y = {y_test_orig.shape if y_test_orig.size > 0 else '0'}")
print(f"Hình dạng đầu vào (Input Shape): {CONFIG['skeleton']['input_shape']}")
print("=" * 50)

## 6. Thiết Kế Kiến Trúc Mạng Transformer Encoder cho Chuỗi Khung Xương

Kiến trúc mạng Transformer được xây dựng chuyên biệt để xử lý chuỗi đặc trưng thời gian - không gian:
* **Linear Feature Projection:** Chiếu vector 51 chiều của mỗi frame lên không gian đặc trưng ẩn $d_{\text{model}} = 64$.
* **Learned Positional Embeddings:** Nhúng thông tin thứ tự thời gian cho 30 bước thời gian, cho phép cơ chế chú ý nhận biết tiến trình vận động (đứng bình thường $\rightarrow$ mất thăng bằng $\rightarrow$ tiếp đất va chạm).
* **Transformer Encoder Blocks (3 khối):**
  - **Multi-Head Self-Attention (MHA):** 4 Attention Heads, với $d_k = 64 / 4 = 16$.
  - **Layer Normalization & Residual Add:** Chuẩn hóa theo lớp với hằng số $\epsilon = 10^{-6}$, kết hợp đường tắt phần dư (Skip Connection).
  - **Feed-Forward Network (FFN):** Hai tầng Dense liên tiếp ($128 \rightarrow 64$) với kích hoạt ReLU và Dropout $0.06$.
* **Phân loại hành vi (Classification Head):**
  - **Global Average Pooling 1D:** Gom tổng hợp thông tin trên toàn bộ 30 khung thời gian thành vector đặc trưng duy nhất.
  - **Dense Layer:** 32 nơ-ron kích hoạt ReLU.
  - **Sigmoid Output Layer:** 1 nơ-ron xuất ra xác suất té ngã $P(\text{fall}) \in [0, 1]$.

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, Dropout, LayerNormalization, MultiHeadAttention,
    GlobalAveragePooling1D, Embedding, Add
)
from tensorflow.keras.optimizers import Adam

# ===== 1. KHỐI TRANSFORMER ENCODER =====
def transformer_encoder_block(inputs, d_model, num_heads, ff_dim, dropout_rate=0.1, name_prefix=""):
    # Cơ chế Self-Attention đa đầu
    attention_output = MultiHeadAttention(
        num_heads=num_heads,
        key_dim=d_model // num_heads,
        dropout=dropout_rate,
        name=f"{name_prefix}_mha"
    )(inputs, inputs, inputs)
    attention_output = Dropout(dropout_rate, name=f"{name_prefix}_mha_dropout")(attention_output)
    out1 = LayerNormalization(epsilon=1e-6, name=f"{name_prefix}_layernorm1")(inputs + attention_output)

    # Mạng Feed-Forward hai tầng
    ffn_output = Dense(ff_dim, activation="relu", name=f"{name_prefix}_ffn_dense1")(out1)
    ffn_output = Dense(d_model, name=f"{name_prefix}_ffn_dense2")(ffn_output)
    ffn_output = Dropout(dropout_rate, name=f"{name_prefix}_ffn_dropout")(ffn_output)
    out2 = LayerNormalization(epsilon=1e-6, name=f"{name_prefix}_layernorm2")(out1 + ffn_output)
    return out2


# ===== 2. TẦNG NHÚNG VỊ TRÍ HỌC ĐƯỢC (LEARNED POSITIONAL EMBEDDING) =====
def get_positional_embedding(seq_len, d_model, name_prefix=""):
    positions = tf.range(start=0, limit=seq_len, delta=1)
    position_embeddings_2d = Embedding(
        input_dim=seq_len, output_dim=d_model, name=f"{name_prefix}_pos_embed"
    )(positions)
    position_embeddings_3d = tf.expand_dims(position_embeddings_2d, axis=0)
    return position_embeddings_3d


# ===== 3. HÀM XÂY DỰNG MÔ HÌNH TRANSFORMER CLASSIFIER HOÀN CHỈNH =====
def create_transformer_classifier(
    input_shape,
    num_encoder_blocks=3,
    d_model=64,
    num_heads=4,
    ff_dim=128,
    projection_dim=None,
    final_dense_units=32,
    dropout_rate=0.06,
    learning_rate=0.0001
):
    timesteps, num_features = input_shape
    if projection_dim is None:
        projection_dim = d_model

    inputs = Input(shape=input_shape, name="input_features")
    
    # 1. Chiếu đặc trưng lên không gian d_model
    projected_x = Dense(projection_dim, name="feature_projection")(inputs)
    
    # 2. Cộng mã hóa vị trí thời gian
    pos_embeddings = get_positional_embedding(timesteps, projection_dim, name_prefix="pos_enc")
    x = Add(name="add_positional_encoding")([projected_x, pos_embeddings])
    x = Dropout(dropout_rate, name="input_dropout_after_pos_enc")(x)

    # 3. Chuỗi các khối Transformer Encoder xếp chồng
    for i in range(num_encoder_blocks):
        x = transformer_encoder_block(
            x, d_model=projection_dim, num_heads=num_heads, ff_dim=ff_dim,
            dropout_rate=dropout_rate, name_prefix=f"encoder_block_{i+1}"
        )

    # 4. Gom tổng hợp và phân loại
    x = GlobalAveragePooling1D(name="global_avg_pooling")(x)
    x = Dropout(dropout_rate, name="dropout_after_pooling")(x)
    x = Dense(final_dense_units, activation="relu", name="final_dense_1")(x)
    x = Dropout(dropout_rate / 2.0, name="dropout_final_dense")(x)
    outputs = Dense(1, activation="sigmoid", name="output_sigmoid")(x)

    model = Model(inputs=inputs, outputs=outputs, name="Fall_Detection_Transformer")

    # Chỉ số F1-Score tùy biến cho Keras
    f1_metric = tf.keras.metrics.F1Score(
        average='macro',
        threshold=0.5,
        name='f1_macro'
    )

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy', f1_metric]
    )
    return model

# Khởi tạo mô hình mẫu để kiểm tra kiến trúc và tổng số tham số
sample_model = create_transformer_classifier(
    input_shape=CONFIG['skeleton']['input_shape'],
    num_encoder_blocks=CONFIG['model']['num_encoder_blocks'],
    d_model=CONFIG['model']['d_model'],
    num_heads=CONFIG['model']['num_heads'],
    ff_dim=CONFIG['model']['ff_dim'],
    projection_dim=CONFIG['model']['projection_dim'],
    final_dense_units=CONFIG['model']['final_dense_units'],
    dropout_rate=CONFIG['model']['dropout_rate'],
    learning_rate=CONFIG['training']['initial_learning_rate']
)
sample_model.summary()

## 7. Đánh Giá Chéo K-Fold (Stratified 5-Fold Cross-Validation)

Để kiểm chứng tính khái quát hóa và độ ổn định của kiến trúc Transformer trên tập dữ liệu chuỗi khung xương, chúng ta thực hiện kiểm thử chéo phân tầng **Stratified 5-Fold Cross-Validation**:
* Mỗi fold giữ nguyên tỉ lệ cân bằng giữa hai lớp `no_fall` và `fall`.
* Dừng sớm (**EarlyStopping**) trên từng fold với `patience = 10` giám sát `val_f1_macro`.
* Tổng hợp trung bình cộng và độ lệch chuẩn của **Loss, Accuracy, F1-Macro**.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from tensorflow.keras.callbacks import EarlyStopping

N_SPLITS = CONFIG['training']['cv_splits']
EPOCHS_CV = CONFIG['training']['cv_epochs']
BATCH_SIZE_CV = CONFIG['training']['cv_batch_size']
PATIENCE_CV = CONFIG['training']['cv_patience']

cv_accuracies, cv_losses, cv_f1_macros = [], [], []

if X_train_orig.shape[0] > 0 and len(np.unique(y_train_orig)) >= 2:
    print(f"\n--- Bắt đầu Đánh Giá Chéo Stratified {N_SPLITS}-Fold Cross-Validation ---")
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=CONFIG['training']['seed'])
    
    for fold_no, (train_idx, val_idx) in enumerate(skf.split(X_train_orig, y_train_orig), start=1):
        print(f"\n>>> Tiến hành Fold {fold_no}/{N_SPLITS} <<<")
        X_cv_train, X_cv_val = X_train_orig[train_idx], X_train_orig[val_idx]
        y_cv_train, y_cv_val = y_train_orig[train_idx].reshape(-1, 1), y_train_orig[val_idx].reshape(-1, 1)

        cv_model = create_transformer_classifier(
            input_shape=CONFIG['skeleton']['input_shape'],
            num_encoder_blocks=CONFIG['model']['num_encoder_blocks'],
            d_model=CONFIG['model']['d_model'],
            num_heads=CONFIG['model']['num_heads'],
            ff_dim=CONFIG['model']['ff_dim'],
            projection_dim=CONFIG['model']['projection_dim'],
            final_dense_units=CONFIG['model']['final_dense_units'],
            dropout_rate=CONFIG['model']['dropout_rate'],
            learning_rate=CONFIG['training']['initial_learning_rate']
        )

        early_stopping_cv = EarlyStopping(
            monitor='val_f1_macro',
            patience=PATIENCE_CV,
            verbose=0,
            mode='max',
            restore_best_weights=True
        )

        history_cv = cv_model.fit(
            X_cv_train, y_cv_train,
            epochs=EPOCHS_CV,
            batch_size=BATCH_SIZE_CV,
            validation_data=(X_cv_val, y_cv_val),
            callbacks=[early_stopping_cv],
            verbose=0
        )

        eval_res = cv_model.evaluate(X_cv_val, y_cv_val, verbose=0)
        cv_losses.append(eval_res[0])
        cv_accuracies.append(eval_res[1])
        cv_f1_macros.append(eval_res[2])
        print(f"  * Kết quả Fold {fold_no}: Loss={eval_res[0]:.4f}, Accuracy={eval_res[1]:.4f}, F1-Macro={eval_res[2]:.4f}")

    print("\n" + "=" * 50)
    print(f"TỔNG KẾT HIỆU NĂNG {N_SPLITS}-FOLD CROSS-VALIDATION:")
    print(f"  * Mean Accuracy : {np.mean(cv_accuracies):.4f} +/- {np.std(cv_accuracies):.4f}")
    print(f"  * Mean Loss     : {np.mean(cv_losses):.4f} +/- {np.std(cv_losses):.4f}")
    print(f"  * Mean F1-Macro : {np.mean(cv_f1_macros):.4f} +/- {np.std(cv_f1_macros):.4f}")
    print("=" * 50)
else:
    print("[Bỏ qua] Dữ liệu huấn luyện chưa đủ hoặc không có đủ 2 lớp để thực hiện Cross-Validation.")

## 8. Huấn Luyện Mô Hình Chính Thức (Final Production Model Training)

Mô hình sản xuất chính thức được huấn luyện trên toàn bộ tập Train (đã tăng cường) với các cơ chế tối ưu:
1. **Lịch trình Suy giảm Tốc độ Học Đa thức (Polynomial Decay Learning Rate Schedule):**
   $$LR(step) = (LR_{\text{initial}} - LR_{\text{end}}) \times \left(1 - \frac{step}{decay\_steps}\right)^{power} + LR_{\text{end}}$$
   * Tốc độ ban đầu: $10^{-4}$, tốc độ kết thúc: $5 \times 10^{-5}$, lũy thừa $power = 0.5$.
   * Cơ chế này giúp mô hình thích nghi nhanh ở các epoch đầu và hội tụ bền vững, tránh hiện tượng dao động quanh cực tiểu địa phương ở các epoch cuối.
2. **Cơ chế Dừng Sớm (EarlyStopping):**
   * Giám sát chỉ số `val_f1_macro` với `patience = 15`.
   * Tự động phục hồi trọng số tốt nhất (`restore_best_weights=True`).

In [ ]:
BATCH_SIZE_FINAL = CONFIG['training']['final_batch_size']
EPOCHS_FINAL = CONFIG['training']['final_epochs']
PATIENCE_FINAL = CONFIG['training']['final_patience']
INITIAL_LR = CONFIG['training']['initial_learning_rate']
END_LR = CONFIG['training']['end_learning_rate']

final_model = None
history_final = None

if X_train_orig.shape[0] > 0:
    y_train_orig_2d = y_train_orig.reshape(-1, 1)
    validation_data_for_fit = None

    if X_val_orig.size > 0 and y_val_orig.size > 0:
        y_val_orig_2d = y_val_orig.reshape(-1, 1)
        validation_data_for_fit = (X_val_orig, y_val_orig_2d)

    print("\n--- Bắt đầu Huấn Luyện Mô Hình Transformer Chính Thức ---")
    final_model = create_transformer_classifier(
        input_shape=CONFIG['skeleton']['input_shape'],
        num_encoder_blocks=CONFIG['model']['num_encoder_blocks'],
        d_model=CONFIG['model']['d_model'],
        num_heads=CONFIG['model']['num_heads'],
        ff_dim=CONFIG['model']['ff_dim'],
        projection_dim=CONFIG['model']['projection_dim'],
        final_dense_units=CONFIG['model']['final_dense_units'],
        dropout_rate=CONFIG['model']['dropout_rate'],
        learning_rate=INITIAL_LR
    )

    # Thiết lập Lịch học Polynomial Decay
    num_train_samples = X_train_orig.shape[0]
    total_decay_steps = (num_train_samples // BATCH_SIZE_FINAL) * EPOCHS_FINAL

    final_lr_schedule = tf.keras.optimizers.schedules.PolynomialDecay(
        initial_learning_rate=INITIAL_LR,
        decay_steps=total_decay_steps,
        end_learning_rate=END_LR,
        power=0.5
    )

    optimizer_scheduled = Adam(learning_rate=final_lr_schedule)
    f1_metric = tf.keras.metrics.F1Score(average='macro', threshold=0.5, name='f1_macro')
    
    final_model.compile(
        optimizer=optimizer_scheduled,
        loss='binary_crossentropy',
        metrics=['accuracy', f1_metric]
    )
    print(f"Đã kích hoạt PolynomialDecay LR: {INITIAL_LR} -> {END_LR} (Total Steps: {total_decay_steps})")

    # Danh sách Callbacks
    callbacks = []
    if validation_data_for_fit is not None:
        early_stopping = EarlyStopping(
            monitor='val_f1_macro',
            patience=PATIENCE_FINAL,
            verbose=1,
            mode='max',
            restore_best_weights=True
        )
        callbacks.append(early_stopping)

    class PrintLearningRate(tf.keras.callbacks.Callback):
        def on_epoch_end(self, epoch, logs=None):
            lr = self.model.optimizer.learning_rate
            if isinstance(lr, tf.keras.optimizers.schedules.LearningRateSchedule):
                lr = lr(self.model.optimizer.iterations)
            try:
                lr = tf.keras.backend.get_value(lr)
            except:
                pass
            print(f" [LR hiện tại: {lr:.7f}]", end="")

    callbacks.append(PrintLearningRate())

    # Thực thi Huấn luyện
    history_final = final_model.fit(
        X_train_orig, y_train_orig_2d,
        epochs=EPOCHS_FINAL,
        batch_size=BATCH_SIZE_FINAL,
        validation_data=validation_data_for_fit,
        callbacks=callbacks,
        verbose=1
    )

    # Đánh giá nhanh trên tập Kiểm Thử
    if X_test_orig.size > 0 and y_test_orig.size > 0:
        y_test_orig_2d = y_test_orig.reshape(-1, 1)
        test_eval = final_model.evaluate(X_test_orig, y_test_orig_2d, verbose=0)
        print(f"\n[Kết quả Test Set] Loss: {test_eval[0]:.4f} | Accuracy: {test_eval[1]:.4f} | F1-Macro: {test_eval[2]:.4f}")
else:
    print("[Lỗi] Tập huấn luyện rỗng. Không thể tiến hành huấn luyện mô hình.")

In [ ]:
import matplotlib.pyplot as plt

# ===== TRỰC QUAN HÓA TIẾN TRÌNH HUẤN LUYỆN (TRAINING HISTORY) =====
if history_final is not None and history_final.history:
    hist = history_final.history
    epochs_range = range(1, len(hist['loss']) + 1)

    plt.figure(figsize=(18, 5))

    # 1. Đồ thị Loss
    plt.subplot(1, 3, 1)
    plt.plot(epochs_range, hist['loss'], label='Train Loss', color='tab:blue', linewidth=2)
    if 'val_loss' in hist:
        plt.plot(epochs_range, hist['val_loss'], label='Val Loss', color='tab:red', linestyle='--', linewidth=2)
    plt.title('Đường Cong Tổn Thất (Loss Curve)', fontsize=12)
    plt.xlabel('Epochs')
    plt.ylabel('Binary Cross-Entropy')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # 2. Đồ thị Độ chính xác (Accuracy)
    plt.subplot(1, 3, 2)
    plt.plot(epochs_range, hist['accuracy'], label='Train Accuracy', color='tab:green', linewidth=2)
    if 'val_accuracy' in hist:
        plt.plot(epochs_range, hist['val_accuracy'], label='Val Accuracy', color='tab:orange', linestyle='--', linewidth=2)
    plt.title('Độ Chính Xác (Accuracy)', fontsize=12)
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # 3. Đồ thị F1-Macro Score
    plt.subplot(1, 3, 3)
    if 'f1_macro' in hist:
        plt.plot(epochs_range, hist['f1_macro'], label='Train F1-Macro', color='tab:purple', linewidth=2)
    if 'val_f1_macro' in hist:
        plt.plot(epochs_range, hist['val_f1_macro'], label='Val F1-Macro', color='tab:brown', linestyle='--', linewidth=2)
    plt.title('Chỉ Số F1-Macro Score', fontsize=12)
    plt.xlabel('Epochs')
    plt.ylabel('F1-Macro')
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("[Lưu ý] Không có lịch sử huấn luyện để trực quan hóa.")

## 9. Đánh Giá Toàn Diện, Tối Ưu Hóa Ngưỡng Kích Hoạt & Phân Tích Lỗi

Trong các bài toán an toàn và y tế, hiện tượng **Bỏ sót té ngã (False Negative - FN)** gây nguy hiểm tính mạng, nghiêm trọng hơn nhiều so với việc cảnh báo nhầm (False Positive - FP). Do đó:
1. **Tìm Ngưỡng Cắt Xác Suất Tối Ưu ($T^*$):** Quét dải ngưỡng $T \in [0.10, 0.90]$ trên tập Validation nhằm tối đa hóa điểm số F1-Score / F2-Score (với F2-Score đặt trọng số gấp đôi cho Recall).
2. **Đánh Giá Đa Chiều:** Báo cáo chi tiết các chỉ số:
   * **Precision (Độ chuẩn xác):** Tỉ lệ cảnh báo ngã đúng trên tổng số lần hệ thống báo ngã.
   * **Recall (Độ nhạy):** Tỉ lệ phát hiện được ca ngã trên thực tế.
   * **F1-Score / F2-Score:** Trung bình điều hòa cân bằng giữa Precision và Recall.
   * **Ma Trận Nhầm Lẫn (Confusion Matrix):** Phân định rõ TP, TN, FP, FN.

In [ ]:
from sklearn.metrics import f1_score, fbeta_score, precision_recall_fscore_support, confusion_matrix
import pandas as pd

# ===== 1. QUÉT NGƯỠNG CẮT TỐI ƯU TRÊN TẬP VALIDATION =====
best_threshold = CONFIG['training']['default_threshold']
best_val_f1 = 0.0

if final_model is not None and X_val_orig.size > 0:
    y_pred_val_probs = final_model.predict(X_val_orig, verbose=0)
    thresholds = np.arange(0.10, 0.91, 0.01)
    
    for thresh in thresholds:
        preds = (y_pred_val_probs > thresh).astype(int)
        score = f1_score(y_val_orig, preds, average='macro')
        if score > best_val_f1:
            best_val_f1 = score
            best_threshold = float(thresh)
            
    print(f"[Tối ưu ngưỡng] Ngưỡng cắt tối ưu xác định trên Validation: {best_threshold:.2f} (F1-Macro = {best_val_f1:.4f})")
else:
    print(f"[Mặc định] Sử dụng ngưỡng chuẩn: {best_threshold:.2f}")


# ===== 2. BÁO CÁO ĐÁNH GIÁ CHI TIẾT TRÊN CÁC TẬP DỮ LIỆU =====
TARGET_NAMES = ['no_fall', 'fall']
CLASS_LABELS = [0, 1]

def evaluate_predictions(y_true, y_probs, threshold, set_name="Test"):
    y_pred = (y_probs > threshold).astype(int).flatten()
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=CLASS_LABELS, zero_division=0
    )
    f2 = fbeta_score(y_true, y_pred, beta=2, labels=CLASS_LABELS, average=None, zero_division=0)
    
    print(f"\n--- BÁO CÁO HIỆU NĂNG TRÊN TẬP {set_name.upper()} (Ngưỡng = {threshold:.2f}) ---")
    print(f"{'Lớp':<12} {'Precision':>10} {'Recall':>10} {'F1-score':>10} {'F2-score':>10} {'Số lượng':>10}")
    print("-" * 66)
    for i, c_name in enumerate(TARGET_NAMES):
        print(f"{c_name:<12} {precision[i]:>10.4f} {recall[i]:>10.4f} {f1[i]:>10.4f} {f2[i]:>10.4f} {support[i]:>10}")
    print("-" * 66)
    
    cm = confusion_matrix(y_true, y_pred, labels=CLASS_LABELS)
    return {
        'precision': precision, 'recall': recall, 'f1': f1, 'f2': f2,
        'confusion_matrix': cm, 'y_pred': y_pred
    }

eval_results_val = None
eval_results_test = None

if final_model is not None:
    if X_val_orig.size > 0:
        y_pred_val_probs = final_model.predict(X_val_orig, verbose=0)
        eval_results_val = evaluate_predictions(y_val_orig, y_pred_val_probs, best_threshold, set_name="Validation")
    
    if X_test_orig.size > 0:
        y_pred_test_probs = final_model.predict(X_test_orig, verbose=0)
        eval_results_test = evaluate_predictions(y_test_orig, y_pred_test_probs, best_threshold, set_name="Test")

In [ ]:
import seaborn as sns

# ===== TRỰC QUAN HÓA MA TRẬN NHẦM LẪN VÀ BARCHART HIỆU NĂNG =====
def plot_evaluation_charts(eval_data, set_name="Test"):
    if eval_data is None:
        return

    cm = eval_data['confusion_matrix']
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 1. Ma Trận Nhầm Lẫn (Confusion Matrix Heatmap)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=TARGET_NAMES, yticklabels=TARGET_NAMES, ax=axes[0]
    )
    axes[0].set_title(f'Ma Trận Nhầm Lẫn (Confusion Matrix) - {set_name}', fontsize=12)
    axes[0].set_xlabel('Nhãn Dự Đoán (Predicted Label)')
    axes[0].set_ylabel('Nhãn Thực Tế (True Label)')

    # 2. Biểu Đồ Cột Chỉ Số Từng Lớp (Metrics Bar Chart)
    metrics_dict = {
        'Precision': eval_data['precision'],
        'Recall': eval_data['recall'],
        'F1-Score': eval_data['f1'],
        'F2-Score': eval_data['f2']
    }
    df_metrics = pd.DataFrame(metrics_dict, index=TARGET_NAMES)
    df_metrics.plot(kind='bar', ax=axes[1], colormap='viridis', width=0.7)
    axes[1].set_title(f'Chỉ Số Đánh Giá Từng Lớp - {set_name}', fontsize=12)
    axes[1].set_ylabel('Điểm số (0 - 1.0)')
    axes[1].set_ylim(0, 1.15)
    axes[1].grid(True, alpha=0.3, axis='y')
    axes[1].tick_params(axis='x', rotation=0)

    for p in axes[1].patches:
        height = p.get_height()
        if height > 0:
            axes[1].annotate(
                f"{height*100:.1f}%",
                (p.get_x() + p.get_width() / 2., height),
                ha='center', va='center', xytext=(0, 6), textcoords='offset points', fontsize=9
            )

    plt.tight_layout()
    plt.show()

# Hiển thị trực quan cho tập Validation và Test
if eval_results_val:
    plot_evaluation_charts(eval_results_val, set_name="Validation")

if eval_results_test:
    plot_evaluation_charts(eval_results_test, set_name="Test")

## 10. Phân Tích Sai Số Chuyên Sâu & Xuất Báo Cáo CSV (Error Analysis)

Nhằm mục đích cải tiến bộ dữ liệu và khắc phục các điểm yếu của mô hình, toàn bộ các ca dự đoán sai được ghi nhận:
* **False Positive (FP - Báo động nhầm):** Người dùng thực hiện sinh hoạt bình thường (nhặt đồ, cúi người nhanh, nằm xuống giường) nhưng mô hình phán đoán nhầm là té ngã.
* **False Negative (FN - Bỏ sót):** Cú ngã thực tế nhưng mô hình không kịp thời nhận diện.

Báo cáo chi tiết được lưu dưới định dạng bảng `error_analysis_report.csv` bao gồm: Tên tệp tin, Nhãn thực tế, Nhãn dự đoán, Xác suất ngã $P(\text{fall})$, Loại lỗi (FP/FN) và Phân vùng tập dữ liệu (`val`/`test`).

In [ ]:
import csv

csv_error_log_filename = CONFIG['paths']['csv_error_log']
error_rows = []

def extract_errors(data_set_name, y_true, y_pred, y_probs, filepaths):
    if len(y_true) != len(y_pred) or len(y_true) != len(filepaths):
        print(f"[Cảnh báo] Dữ liệu tập {data_set_name} không khớp kích thước. Bỏ qua ghi log.")
        return

    fp_indices = np.where((y_pred == 1) & (y_true == 0))[0]
    fn_indices = np.where((y_pred == 0) & (y_true == 1))[0]

    print(f"[Phân tích lỗi tập {data_set_name}] Tổng số mẫu: {len(y_true)} | False Positives (FP): {len(fp_indices)} | False Negatives (FN): {len(fn_indices)}")

    for idx in fp_indices:
        prob = float(y_probs[idx][0]) if hasattr(y_probs[idx], '__len__') else float(y_probs[idx])
        error_rows.append({
            'Tập dữ liệu': data_set_name,
            'Tên tệp': Path(filepaths[idx]).name,
            'Đường dẫn đầy đủ': filepaths[idx],
            'Nhãn thực tế': 'no_fall (0)',
            'Dự đoán': 'fall (1)',
            'Xác suất P(fall)': f"{prob:.4f}",
            'Loại sai số': 'FP (Báo Giả)'
        })

    for idx in fn_indices:
        prob = float(y_probs[idx][0]) if hasattr(y_probs[idx], '__len__') else float(y_probs[idx])
        error_rows.append({
            'Tập dữ liệu': data_set_name,
            'Tên tệp': Path(filepaths[idx]).name,
            'Đường dẫn đầy đủ': filepaths[idx],
            'Nhãn thực tế': 'fall (1)',
            'Dự đoán': 'no_fall (0)',
            'Xác suất P(fall)': f"{prob:.4f}",
            'Loại sai số': 'FN (Bỏ Sót)'
        })

# Thu thập lỗi từ tập Val và Test
if 'eval_results_val' in locals() and eval_results_val and len(val_npy_filepaths) > 0:
    extract_errors('Validation', y_val_orig, eval_results_val['y_pred'], y_pred_val_probs, val_npy_filepaths)

if 'eval_results_test' in locals() and eval_results_test and len(test_npy_filepaths) > 0:
    extract_errors('Test', y_test_orig, eval_results_test['y_pred'], y_pred_test_probs, test_npy_filepaths)

# Lưu báo cáo vào tệp CSV
if error_rows:
    fieldnames = ['#', 'Tập dữ liệu', 'Tên tệp', 'Nhãn thực tế', 'Dự đoán', 'Xác suất P(fall)', 'Loại sai số', 'Đường dẫn đầy đủ']
    with open(csv_error_log_filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for i, row in enumerate(error_rows, start=1):
            writer.writerow({'#': i, **row})
    print(f"\n-> Báo cáo phân tích sai số đã được xuất thành công vào: {csv_error_log_filename}")

    # Xem nhanh 5 trường hợp lỗi đầu tiên
    df_err = pd.read_csv(csv_error_log_filename)
    print(f"Tổng số trường hợp dự đoán sai cần tối ưu: {len(df_err)}")
    display(df_err.head(10)) if 'display' in locals() else print(df_err.head())
else:
    print(f"\n-> Tuyệt vời! Không phát hiện trường hợp sai số nào (FP hoặc FN) hoặc chưa có dữ liệu đánh giá.")

## 11. Đóng Gói & Tối Ưu Hóa Biên Cho Thiết Bị IoT (TensorFlow Lite Export)

Để triển khai thuật toán trên các phần cứng nhúng biên (Edge AI Devices như Raspberry Pi 4 / 5, Jetson Nano, Orange Pi):
1. **Xuất Định Dạng Keras SavedModel:** Đóng gói toàn bộ kiến trúc đồ thị và trọng số đã tối ưu.
2. **Chuyển Đổi Sang TensorFlow Lite (`.tflite`):** Tối ưu hóa kích thước đồ thị tĩnh, giảm tải dung lượng mô hình chỉ còn **~313 KB**.
3. **Thử Nghiệm Suy Luận Dummy (Latency Benchmark):** Khởi tạo `tf.lite.Interpreter`, nạp mẫu chuỗi giả lập $(1, 30, 51)$ và đo đạc độ trễ phản hồi trung bình (milliseconds / lần suy luận) trên CPU.

In [ ]:
saved_model_dir = CONFIG['paths']['export_saved_model']
tflite_path = CONFIG['paths']['export_tflite']

if final_model is not None and isinstance(final_model, tf.keras.Model):
    print("=" * 60)
    print("BẮT ĐẦU QUY TRÌNH XUẤT MÔ HÌNH SANG TENSORFLOW LITE")
    print("=" * 60)

    # 1. Lưu mô hình ở định dạng SavedModel
    print(f"1. Xuất Keras SavedModel tại: {saved_model_dir}")
    final_model.export(saved_model_dir)

    # 2. Chuyển đổi sang TensorFlow Lite
    print(f"2. Chuyển đổi SavedModel -> {tflite_path}...")
    converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
    tflite_content = converter.convert()

    with open(tflite_path, 'wb') as f:
        f.write(tflite_content)

    file_size_kb = len(tflite_content) / 1024.0
    print(f"   -> Chuyển đổi TFLite thành công!")
    print(f"   -> Đường dẫn tệp tin: {os.path.abspath(tflite_path)}")
    print(f"   -> Kích thước tệp   : {file_size_kb:.2f} KB (~ {file_size_kb/1024:.2f} MB)")

    # 3. Kiểm thử suy luận với Interpreter (Benchmark Latency trên CPU)
    print("\n3. Kiểm định suy luận thực nghiệm trên môi trường nhúng:")
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    print(f"   - Input Shape  : {input_details[0]['shape']} ({input_details[0]['dtype']})")
    print(f"   - Output Shape : {output_details[0]['shape']} ({output_details[0]['dtype']})")

    # Mẫu dữ liệu giả lập (Dummy Input)
    dummy_input = np.random.randn(*input_details[0]['shape']).astype(np.float32)
    interpreter.set_tensor(input_details[0]['index'], dummy_input)
    interpreter.invoke()
    dummy_output = interpreter.get_tensor(output_details[0]['index'])
    print(f"   - Xác suất ngã mẫu giả định: P(fall) = {dummy_output[0][0]:.4f}")

    # Đo độ trễ suy luận trung bình qua 100 lần lặp
    latencies = []
    for _ in range(100):
        t0 = time.time()
        interpreter.set_tensor(input_details[0]['index'], dummy_input)
        interpreter.invoke()
        _ = interpreter.get_tensor(output_details[0]['index'])
        latencies.append((time.time() - t0) * 1000.0)

    avg_latency = np.mean(latencies)
    p95_latency = np.percentile(latencies, 95)
    print(f"   - Độ trễ suy luận trung bình: {avg_latency:.2f} ms / lần suy luận")
    print(f"   - Độ trễ P95               : {p95_latency:.2f} ms / lần suy luận")
    print(f"   - Tốc độ suy luận ước tính : ~{1000.0 / avg_latency:.1f} FPS trên CPU")
    print("=" * 60)
    print("HOÀN TẤT THÍ NGHIỆM VÀ ĐÓNG GÓI MÔ HÌNH THÀNH CÔNG!")
else:
    print("[Lỗi] final_model chưa được khởi tạo hoặc huấn luyện.")